In [23]:
# langchain框架提供了如下组件：
# Loader:文件加载器
# text splitter:文件分块器
# embedding models:嵌入模型
# vectorstores:向量存储
# retrievers:检索器
# prompt:提示词
# chain:链将各个组件按特定的顺序串起来

In [1]:
# 导入提示词模板
from langchain_core.prompts import PromptTemplate

In [13]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter # 分块：递归分块，字符分块
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path

In [64]:
# 设置模型
LLM = ChatOpenAI(
    base_url="https://api.deepseek.com/v1",
    api_key="sk-0e90c3db27dd481f8542e1faf5cbf39e",
    model="deepseek-chat"  # 明确指定模型
)
# 嵌入模型
embedding_model = HuggingFaceEmbeddings(model_name=r"D:\PycharmProjects\pyjupyter\深度学习\深度学习代码\RAG检索增强生成\model\BAAI\bge-large-zh-v1___5")

In [77]:
# 设计数据处理(文本加载、分块、存储、检索)
file_dir = Path("knnowledge")
# 文本分块
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 100)
# 存储
vector_store = Chroma(embedding_function = embedding_model,persist_directory=r"D:\PycharmProjects\pyjupyter\深度学习\深度学习代码\RAG检索增强生成\chromadb")
# 检索
retriever = vector_store.as_retriever(search_kwargs={"k":5})# 返回top-k的文档

# 提示词模板
prompt_template = PromptTemplate.from_template("""请根据上下文来回答问题，如果上下文信息不在足以回答问题，请直接说“根据上下文信息，无法回答”
        上下文：{context}
        问题：{question}
""")

In [78]:
# 编排langchain的链
chain = {"question":RunnablePassthrough()} |RunnablePassthrough.assign(context=itemgetter("question")| retriever) | prompt_template | LLM | StrOutputParser() 

In [79]:
print(chain.invoke("告诉我关于无人机的知识"))

根据上下文信息，无法回答。
